# Trending news for your watchlist and entities of interest

**Powered by [Bigdata.com](https://bigdata.com)** · [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bigdata-com/bigdata-trending-news/blob/main/notebooks/01_watchlist_trending.ipynb)

**The question:** *"Which stories about my companies and the entities I follow are trending right now?"*

"What's the news on X" is a search problem (Bigdata.com fast search answers it). This notebook answers a
different question: **what is trending**, meaning the stories that many outlets are writing about *right now*,
for entities that are unusually loud compared with their normal level of coverage.

### What "trending" means here

Bigdata.com does **not** capture reader engagement (clicks, reads, shares). Trending is measured from
**publishing momentum**, which is a strong proxy for attention:

| Signal | Question it answers | Measured from |
|---|---|---|
| **Breadth** | How many articles and *distinct outlets* are covering the **same story**? (the "engagement around similar articles") | articles grouped into stories |
| **Acceleration** | Is this entity covered more than usual right now? (`spike = now ÷ normal`) | `/v1/search/volume` |
| **Recency** | How fresh is the latest coverage? | article timestamps |

A story needs **at least 2 independent outlets** to count as trending. Stories with strong positive or negative
sentiment from credible outlets that have *not yet* built wide coverage are shown separately as **Emerging**.

> If you have your own click or read data, it can be blended in later through the
> `external_engagement_score` hook in `trending/rank.py` (default 0).

### What this notebook does

| Step | What happens | Uses an AI model? |
|---|---|---|
| 1. Resolve | Tickers and names → Bigdata.com entity IDs (Knowledge Graph) | No |
| 2. Momentum | Coverage now vs. the entity's normal baseline, number of outlets, sentiment | No |
| 3. Fetch | Pull the current window's news articles for each entity | No |
| 4. Group | Group articles that report the **same event** into one story | Yes (falls back to text similarity) |
| 5. Enrich | Story title, one-line summary, direction, driver, materiality | Yes (optional) |
| 6. Rank | Trend score; separate **Emerging** list for strong-sentiment early signals | No |
| 7. Report | Self-contained HTML report + CSV and an Excel workbook in `outputs/` | No |

All counts, momentum figures and sentiment values are computed from Bigdata.com data. The AI model only groups,
titles and summarises articles. Run the cells top to bottom (about 2 minutes).
Each section shows its typical duration, and every code cell prints how long it took (`⏱`).

### Running in Google Colab

Click the badge above, then run the cells top to bottom. The next cell prepares Colab (it does nothing anywhere else):
it downloads the code, installs the packages and reads your API keys from Colab **Secrets** (🔑 icon in the left
sidebar). Add `BIGDATA_API_KEY` and `OPENAI_API_KEY` there
and switch on *Notebook access*. A `BIGDATA_API_KEY` is required.

In [1]:
# 0. Google Colab only: download the code, install packages, load API keys from Colab Secrets
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    from google.colab import userdata

    def secret(name):
        try:
            return userdata.get(name)
        except Exception:  # secret missing or notebook access not granted
            return None

    repo, target = "Bigdata-com/bigdata-trending-news", "/content/bigdata-trending-news"
    if not os.path.exists(target):
        token = secret("GITHUB_TOKEN")   # optional: only needed for a private fork
        auth = f"x-access-token:{token}@" if token else ""
        subprocess.run(["git", "clone", "-q", f"https://{auth}github.com/{repo}.git", target], check=True)
        subprocess.run(["git", "-C", target, "remote", "set-url", "origin", f"https://github.com/{repo}.git"])
    # Colab already has pandas and Jupyter; install only what is missing.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "python-dotenv", "jinja2",
                    "openai", "openpyxl"], check=True)
    os.chdir(f"{target}/notebooks")
    for name in ["BIGDATA_API_KEY", "OPENAI_API_KEY", "LLM_MODEL"]:
        if value := secret(name):
            os.environ[name] = value
    print("Colab ready. Keys found:", [n for n in ["BIGDATA_API_KEY", "OPENAI_API_KEY"]
                                      if os.environ.get(n)] or "none")
else:
    print("Not running in Colab: nothing to do.")

Not running in Colab: nothing to do.


In [2]:
# Setup: make the shared `trending` package importable, tidy pandas output, time every cell.
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from trending.timing import cell_timer
cell_timer.start()   # from here on, every cell ends with "⏱ <seconds>"; the last cell shows a summary

import pandas as pd
from IPython.display import HTML, display

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

from trending import config as cfg
from trending.pipeline import start, build_stories, finish, llm_label
from trending.resolve import resolve_all, entity_table
from trending.momentum import compute_momentum, momentum_table
from trending.rank import rank_trending, rank_emerging
from trending import render

⏱ 0.0 s


## 1. Configuration

Edit the lists below to change what is tracked. Everything else (weights, thresholds, windows) lives in
`trending/config.py` and can also be overridden here, e.g. `cfg.SENTIMENT_EXTREME = 0.4`.

* **API key:** `BIGDATA_API_KEY` is required.
* **LLM:** OpenAI GPT 5.6 (`OPENAI_API_KEY`). Set `cfg.USE_LLM = False` to run without any AI model.

> ⏱ **Typical time:** under 1 s

In [3]:
# 1. Configuration
WATCHLIST = ["AAPL", "TSLA", "NVDA", "MSFT", "AMZN", "META", "GOOGL"]   # tickers
ENTITIES_OF_INTEREST = ["OpenAI", "Federal Reserve"]                    # free text, resolved via the Knowledge Graph

cfg.WINDOW = "1D"          # "1D" = last 24h vs prior 7 days  |  "1W" = last 7 days vs prior 28 days
cfg.USE_LLM = True         # False = lexical story grouping, no AI model
cfg.LLM_MODEL = "gpt-5.6"
# Emerging list: |mean chunk sentiment| >= 0.5 (see README, "Tuning")
cfg.SENTIMENT_MODE = "mean"
cfg.SENTIMENT_EXTREME = 0.5

client, llm, win = start("watchlist")

Window 1D: 2026-11-08 19:00 → 2026-11-09 19:00 UTC
LLM: LLM(provider='openai', model='gpt-5.6')
⏱ 0.0 s


## 2. Resolve entities (Knowledge Graph)

Each ticker or name is mapped to a Bigdata.com entity ID. IDs only ever come from the Knowledge Graph (never
invented). When a name matches several entities, the candidates are listed and the one with the most news
coverage over the past week is chosen (→). Check this table: it is what everything below is filtered on.

> ⏱ **Typical time:** under 1 s (a few seconds for names not seen before)

In [4]:
# 2. Resolve entities
entities = resolve_all(client, tickers=WATCHLIST, free_text=ENTITIES_OF_INTEREST, as_of=win.as_of)
entity_table(entities)

,input,label,entity_id,name,kg_type,kg_category,mode,how
0,AAPL,AAPL,D8442A,Apple Inc.,PUBLIC,companies,entity,KG ticker match (matches verified default)
1,TSLA,TSLA,DD3BB1,Tesla Inc.,PUBLIC,companies,entity,KG ticker match (matches verified default)
2,NVDA,NVDA,E09E2B,NVIDIA Corp.,PUBLIC,companies,entity,KG ticker match (matches verified default)
3,MSFT,MSFT,228D42,Microsoft Corp.,PUBLIC,companies,entity,KG ticker match (matches verified default)
4,AMZN,AMZN,0157B1,Amazon.com Inc.,PUBLIC,companies,entity,KG ticker match (matches verified default)
5,META,META,12E454,Meta Platforms Inc.,PUBLIC,companies,entity,KG ticker match (matches verified default)
6,GOOGL,GOOGL,4A6F00,Alphabet Inc.,PUBLIC,companies,entity,KG ticker match (matches verified default)
7,OpenAI,OpenAI,UWCVKI,OpenAI Foundation,PRIVATE,companies,entity,KG companies: best name match with most news
8,Federal Reserve,Federal Reserve,F38FC4,United States Federal Reserve,Central Bank,organizations,entity,KG organizations: best name match with most news


⏱ 0.0 s


## 3. Momentum: which of my names are unusually loud today?

For each entity, two calls to `/v1/search/volume` (news only):

* **current_docs**: documents in the current window; **distinct_sources**: how many outlets published them.
* **baseline_per_period**: the entity's normal volume for a window of the same length (average of the baseline days).
* **spike** = (current_docs + 1) / (baseline_per_period + 1). Above 1 means louder than usual.
* **sentiment**: Bigdata.com's average sentiment (-1 to +1); **trend**: daily document counts for the last 8 days
  (the last day is still in progress).

> ⏱ **Typical time:** about 3 s

In [5]:
# 3. Momentum
def show_momentum(df):
    """Momentum table with a bar on the spike column (no extra dependencies)."""
    return (df.style
              .format({"spike": "{:.2f}×", "baseline_per_period": "{:,.0f}", "current_docs": "{:,}",
                       "sentiment": "{:+.2f}"}, na_rep="–")
              .bar(subset=["spike"], color="#f0a35e", vmin=0)
              .hide(axis="index"))

momentum = compute_momentum(client, entities, win)
show_momentum(momentum_table(momentum, entities))

entity,name,spike,current_docs,baseline_per_period,distinct_sources,sentiment,trend,mode
NVDA,NVIDIA Corp.,13.90×,"5,668",407,1000,-0.41,▁▁▁▁▁▁▂█,entity
Federal Reserve,United States Federal Reserve,1.38×,247,179,98,-0.14,▃▁▂▆█▂▁▄,entity
TSLA,Tesla Inc.,0.95×,363,384,206,-0.32,▁▁▂▁█▁▁▂,entity
GOOGL,Alphabet Inc.,0.81×,623,772,276,-0.08,▃▇█▇▄▃▁▃,entity
META,Meta Platforms Inc.,0.78×,189,243,119,-0.18,▆▅▄█▄▁▁▃,entity
AAPL,Apple Inc.,0.58×,473,812,167,-0.01,▅█▃▂▂▁▁▁,entity
MSFT,Microsoft Corp.,0.58×,163,284,98,-0.05,▃▃▃▅█▁▁▂,entity
AMZN,Amazon.com Inc.,0.55×,167,306,97,+0.07,█▃▅▇▇▂▁▂,entity
OpenAI,OpenAI Foundation,0.36×,119,334,77,-0.24,▅▅▄▇█▂▁▁,entity


⏱ 3.7 s


## 4. Find the trending stories

Fetch the current window's articles → group articles reporting the same event into stories → enrich the leading
stories → score. Progress is printed for each step. This is the longest step: most of it is the AI model.

> ⏱ **Typical time:** about 1-2 min

In [6]:
# 4. Fetch, group, enrich and rank stories
stories, articles, _ = build_stories(client, llm, entities, win, momentum)
trending = rank_trending(stories)

cols = ["rank", "title", "entities", "articles", "sources", "spike", "direction", "driver", "materiality",
        "avg_sentiment", "trend_score"]
trending_df = render.stories_frame(trending, "trending")
trending_df[cols]

Step 3  fetched 1764 unique articles: AAPL 250, TSLA 218, NVDA 329, MSFT 128, AMZN 160, META 151, GOOGL 256, OpenAI 112, Federal Reserve 160  (7s)
Step 4  grouped into 622 stories, 200 covered by ≥2 sources  (75s)
        merged 2 duplicate stories among the top 60
Step 5  enriched 60 stories (title, summary, direction, driver, materiality)  (92s)
        merged 25 more duplicates by title and summary  (97s)
Step 6  scored all stories


,rank,title,entities,articles,sources,spike,direction,driver,materiality,avg_sentiment,trend_score
0,1,Nvidia Shares Tumble as Board Investigates CEO's Alleged Spy Relationship,NVDA,239,54,13.90,down,CEO investigation,5.0,-0.663,3.5037
1,2,Tesla Plans Cybertruck Exit and Deposit Refunds Before 2027,TSLA,138,97,0.95,mixed,Cybertruck production exit,4.0,-0.555,2.7925
2,3,Google Launches Two New AI Chips for Enterprise Workloads,GOOGL,95,56,0.81,up,AI chip launch,4.0,0.233,2.5069
3,4,Nvidia Partners on South Korea Sovereign AI Infrastructure,NVDA,11,9,13.90,up,Korea AI partnership,3.0,0.522,2.1063
4,5,Apple Reportedly Accelerates Plans for More Agentic Siri,AAPL,42,30,0.58,up,Siri AI acceleration,4.0,-0.187,2.0305
5,6,Meta Expands AI Age Checks for Teen Accounts Globally,META,30,30,0.78,mixed,Teen safety rollout,3.0,-0.307,1.9820
6,7,Google warns Australian bill could restrict teen access to Gmail,GOOGL,25,25,0.81,down,online safety regulation,3.0,-0.611,1.8980
7,8,Meta and Amazon shares jump after JPMorgan AI endorsements,"META, AMZN",31,17,0.78,up,analyst endorsements,3.0,0.218,1.8596
8,9,December Fed rate-hike expectations rise after strong jobs report,Federal Reserve,19,15,1.38,n/a,rate hike expectations,5.0,-0.199,1.8074
9,10,Democratic Senate control strengthens safeguards for Federal Reserve independence,Federal Reserve,17,15,1.38,up,Senate election shift,4.0,-0.385,1.7991


⏱ 1 min 37 s


## 5. Emerging signals: strong sentiment, limited coverage so far

**Trending** answers *"what is everyone writing about?"* A strongly positive or negative story that only one or two
credible outlets have covered so far will not rank there yet, so it is captured here instead.

A story qualifies when its Bigdata.com sentiment is extreme (|sentiment| ≥ `SENTIMENT_EXTREME`), it comes from a
top-tier outlet (source rank 1-2) if it has a single source, it is inside the current window, and (with the AI model
on) its materiality is 4 or 5. It never repeats a Trending story.
`emerging score = |sentiment| × materiality/5 × recency`.

> ⏱ **Typical time:** under 1 s

In [7]:
# 5. Emerging signals
emerging = rank_emerging(stories, trending, win, use_llm=llm is not None)

emerging_df = render.stories_frame(emerging, "emerging")
emerging_df[["rank", "title", "entities", "sentiment_signal", "sources", "best_source_rank",
             "representative_source", "driver", "materiality", "emerging_score"]]

,rank,title,entities,sentiment_signal,sources,best_source_rank,representative_source,driver,materiality,emerging_score
0,1,Georgia Residents Sue OpenAI Over $20 Billion Data Center,OpenAI,-0.714,4,2,WJCL-TV,data center lawsuit,4,0.4794
1,2,Nvidia Earnings Loom After Sharp Monthly Share Decline,NVDA,-0.740,1,2,Yahoo! Finance,earnings overhang,4,0.4437


⏱ 0.0 s


## 6. The report

The report below is also saved as a single self-contained HTML file in `outputs/` (open it in any browser or attach
it to an email), together with `*_trending.csv` and `*_emerging.csv`. The next cell writes an Excel workbook with
sheets Trending, Breaking, and Audit. Click **Show all … articles** under a story to
see every article in it, with links.

> ⏱ **Typical time:** under 1 s

In [8]:
# 6. Build and save the report
ctx = render.ReportContext(win=win, momentum=momentum, names={e.key: e.name for e in entities})
scope = f"{len(WATCHLIST)} watchlist names + " + ", ".join(ENTITIES_OF_INTEREST)
TITLE = "Trending news: watchlist & entities of interest"

report = render.watchlist_report(title=TITLE, scope=scope, ctx=ctx, trending=trending, emerging=emerging,
                                 llm_label=llm_label(llm))
paths = render.save_outputs(report, TITLE, {"trending": trending_df, "emerging": emerging_df},
                            "watchlist", day=win.as_of)
for kind, p in paths.items():
    print(f"saved {kind:<9} → {os.path.relpath(p, ROOT)}")
display(HTML(report))

saved html      → outputs/watchlist_trending_2026-11-09.html
saved trending  → outputs/watchlist_2026-11-09_trending.csv
saved emerging  → outputs/watchlist_2026-11-09_emerging.csv


⏱ 0.2 s


In [9]:
# Excel workbook: Trending, Breaking, and Audit
import importlib

importlib.reload(render)  # pick up save_watchlist_excel if this kernel imported render earlier
xlsx = render.save_watchlist_excel(
    trending=trending,
    emerging=emerging,
    stories=stories,
    articles=articles,
    win=win,
    use_llm=llm is not None,
)
print(f"saved {'excel':<9} → {os.path.relpath(xlsx, ROOT)}")

saved excel     → outputs/watchlist_trending_2026-11-09.xlsx
⏱ 2.5 s


In [10]:
# Record usage and prune the local response cache
finish(client, llm)

Bigdata.com query units used this session: 0.0, LLM calls 46 (457 s summed over parallel calls, 0 from cache)
Local cache: fixtures/watchlist/ (removed 145 stale files).
⏱ 0.0 s


## Run time

Every code cell above ends with `⏱`: how long it took in this run. The table adds them up per step. Live runs are
dominated by the AI steps (grouping and enrichment) and by Bigdata.com calls.

In [11]:
# Run time: how long each step took in this run
cell_timer.summary()

,step,time,share,seconds
0,Setup,0.0 s,0%,0.0
1,1. Configuration,0.0 s,0%,0.0
2,2. Resolve entities,0.0 s,0%,0.0
3,3. Momentum,3.7 s,4%,3.7
4,"4. Fetch, group, enrich and rank stories",1 min 37 s,94%,97.1
5,5. Emerging signals,0.0 s,0%,0.0
6,6. Build and save the report,0.2 s,0%,0.2
7,"Excel workbook: Trending, Breaking, and Audit",2.5 s,2%,2.5
8,Record usage and prune the local response cache,0.0 s,0%,0.0
9,Total,1 min 44 s,100%,103.5


## 7. How to adapt

| To… | Change |
|---|---|
| Track other companies | Edit `WATCHLIST` (tickers) or `ENTITIES_OF_INTEREST` (any name: companies, organisations, people, places, concepts) |
| Look at the past week instead of the past day | `cfg.WINDOW = "1W"` (last 7 days vs. prior 28 days) |
| Re-run a specific moment | `cfg.AS_OF = "2026-10-13T18:40:00Z"` |
| Change how stories are ranked | `cfg.WEIGHTS` (articles, sources, spike, recency, materiality) |
| Require more outlets before a story trends | `cfg.TRENDING_MIN_SOURCES` (default 2) |
| Make Emerging stricter / looser | `cfg.SENTIMENT_EXTREME`, `cfg.SENTIMENT_MODE` (`"mean"` or `"max_abs"`), `cfg.EMERGING_MAX_SOURCE_RANK`, `cfg.EMERGING_MIN_MATERIALITY` |
| Include articles that only *mention* an entity | `cfg.SEARCH_IN = "ALL"` (default `"HEADLINE"` = articles *about* the entity) |
| Retrieve more articles per entity | `cfg.FETCH_SLICES = 2` (or more), at a higher API cost |
| Blend in your own click / read data | implement `external_engagement_score(story)` in `trending/rank.py` |
| Run without an AI model | `cfg.USE_LLM = False` |

See `README.md` for the full list of settings and the limitations.